# E-Commerce Order, Refund & Exception Handling: a Multi-Agent Workflow with CrewAI

**Week 12 Graded Mini Project, Use Case 2**

ShopKart (a fictional Indian e-commerce store) receives customer messages about delayed deliveries,
refunds, damaged products and returns. This notebook builds a **goal-oriented multi-agent workflow**
with four collaborating agents:

| # | Agent | Job | Hands off to |
|---|---|---|---|
| 1 | **Order Issue Identification Agent** | Understand the message, find the order, classify the issue | Agent 2 |
| 2 | **Policy Interpretation Agent** | Apply ShopKart's refund/return policy to that issue | Agent 3 |
| 3 | **Resolution Recommendation Agent** | Choose the best allowed resolution and draft a reply | Agent 4 |
| 4 | **Escalation Agent** | Decide whether a human must approve, and route it | Final answer |

**Goal:** fast resolutions while enforcing business policies consistently.

**This is the standalone version** of the project: the same 4 agents, policy rules and escalation rules
as the Docker microservices version, but everything lives in one notebook, with no Docker, MCP server or
frontend. Use it to learn how the crew works step by step, or to submit in Vocareum.

**How to use this notebook:** run the cells top to bottom. Each step starts with a short explanation.
Steps 2–6 need **no API key** (they are plain Python and object definitions), so you can study them first.

---
## Step 0: Check Python and install the packages (run once)

CrewAI needs **Python 3.10 to 3.13**. The first cell tells you which Python this notebook is using.

The second cell installs the packages **into that same Python**. `%pip` (with the percent sign) always
installs into the notebook's own kernel, which avoids the classic *"I installed it but `import` still
fails"* problem. After it finishes, **restart the kernel** (the "Restart" button at the top of the
notebook), then continue from Step 1. You only need to do this once.

In [ ]:
import sys
print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    print("WARNING: CrewAI needs Python 3.10-3.13. Select a different kernel (see README).")
elif ".venv" not in sys.executable:
    print("Note: this is not the project's .venv. That works, but the README's .venv setup is cleaner.")
else:
    print("OK: using the project's virtual environment.")

In [ ]:
%pip install -q crewai==1.15.28 python-dotenv pandas ipykernel

---
## Step 1: Setup: imports, API key and the LLM

The **LLM** is the "brain" every agent uses. We read the model name from `.env`, so you can switch
models without touching code.

`temperature=0` makes answers as consistent as possible, which matters because the same complaint
should always get the same policy decision.

In [ ]:
import os
import json
from datetime import date
from typing import List, Literal, Optional

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

import crewai
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()  # finds the project's .env (same settings as the Docker version)

if os.getenv("MODEL_TYPE", "openai").lower() == "ollama":
    MODEL = "ollama/" + os.getenv("OLLAMA_MODEL", "llama3.1")
    # outside Docker, Ollama runs on localhost rather than host.docker.internal
    base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").replace("host.docker.internal", "localhost")
    llm = LLM(model=MODEL, base_url=base_url, temperature=0)
else:
    MODEL = "openai/" + os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    # Vocareum course keys need OPENAI_BASE_URL=https://openai.vocareum.com/v1 ; personal keys need none
    llm = LLM(model=MODEL, api_key=os.getenv("OPENAI_API_KEY"),
              base_url=os.getenv("OPENAI_BASE_URL") or None, temperature=0)

print("CrewAI version :", crewai.__version__)
print("Model          :", MODEL)
print("Base URL       :", os.getenv("OPENAI_BASE_URL") or "default")
print("API key loaded :", bool(os.getenv("OPENAI_API_KEY")))

---
## Step 2: The knowledge layer (no RAG, no AI)

The brief says RAG is **not required**. We simulate the company's knowledge with:
1. a **mock order database** (a Python dictionary), and
2. the **refund & return policy**, written twice: as text a human can read, and as Python rules a computer can apply.

**Design decision:** anything involving money, deadlines or thresholds is computed by plain Python, never
guessed by the LLM. LLMs are good at understanding language and writing replies; they are bad at
reliably counting days or comparing ₹10,000 against ₹9,999. Splitting the work this way is what makes the
policy "enforced consistently".

We also fix "today" to a constant date so the results are the same whenever you run the notebook.

In [ ]:
REFERENCE_DATE = date(2026, 10, 17)   # "today" for every calculation (keeps results reproducible)

ORDERS = {
    "ORD1001": {"customer": "Rahul Mehta",  "tier": "Silver",  "item": "Wireless Headphones",    "category": "electronics", "price_inr": 2499,  "shipping_fee_inr": 0,  "status": "DELIVERED",  "delivered_on": "2026-10-14", "expected_by": "2026-10-14", "payment": "UPI",  "refund_claims_90d": 0},
    "ORD1002": {"customer": "Anita Rao",    "tier": "Regular", "item": "Cotton Kurta",           "category": "apparel",     "price_inr": 1299,  "shipping_fee_inr": 49, "status": "DELIVERED",  "delivered_on": "2026-10-05", "expected_by": "2026-10-05", "payment": "Card", "refund_claims_90d": 0},
    "ORD1003": {"customer": "Vikram Singh", "tier": "Regular", "item": "Smart Watch",            "category": "electronics", "price_inr": 3999,  "shipping_fee_inr": 99, "status": "IN_TRANSIT", "delivered_on": None,         "expected_by": "2026-10-12", "payment": "Card", "refund_claims_90d": 0},
    "ORD1004": {"customer": "Sneha Iyer",   "tier": "Gold",    "item": "14-inch Laptop",         "category": "electronics", "price_inr": 62000, "shipping_fee_inr": 0,  "status": "DELIVERED",  "delivered_on": "2026-10-15", "expected_by": "2026-10-15", "payment": "Card", "refund_claims_90d": 0},
    "ORD1005": {"customer": "Arjun Das",    "tier": "Regular", "item": "Smartphone",             "category": "electronics", "price_inr": 18999, "shipping_fee_inr": 0,  "status": "DELIVERED",  "delivered_on": "2026-10-13", "expected_by": "2026-10-13", "payment": "UPI",  "refund_claims_90d": 3},
    "ORD1006": {"customer": "Meera Nair",   "tier": "Silver",  "item": "20000mAh Power Bank",    "category": "electronics", "price_inr": 1499,  "shipping_fee_inr": 0,  "status": "DELIVERED",  "delivered_on": "2026-10-10", "expected_by": "2026-10-10", "payment": "COD",  "refund_claims_90d": 0},
    "ORD1007": {"customer": "Karan Gupta",  "tier": "Regular", "item": "Organic Fruit Basket",   "category": "grocery",     "price_inr": 850,   "shipping_fee_inr": 40, "status": "DELIVERED",  "delivered_on": "2026-10-16", "expected_by": "2026-10-16", "payment": "UPI",  "refund_claims_90d": 0},
    "ORD1008": {"customer": "Pooja Sharma", "tier": "Gold",    "item": "Running Shoes",          "category": "apparel",     "price_inr": 4599,  "shipping_fee_inr": 0,  "status": "DELIVERED",  "delivered_on": "2026-09-15", "expected_by": "2026-09-15", "payment": "Card", "refund_claims_90d": 0},
    "ORD1009": {"customer": "Imran Khan",   "tier": "Regular", "item": "Data Science Handbook",  "category": "books",       "price_inr": 899,   "shipping_fee_inr": 40, "status": "DELIVERED",  "delivered_on": "2026-10-14", "expected_by": "2026-10-14", "payment": "Card", "refund_claims_90d": 1},
}

# The human-readable policy. Agents read this text in their instructions.
REFUND_POLICY_TEXT = """
SHOPKART REFUND, RETURN & EXCEPTION POLICY (v1.0)

R1  Damaged / defective item: full refund or free replacement if reported within 7 days of delivery
    (within 2 days for grocery). Photo evidence is required for items priced above Rs 5,000.
R2  Wrong item delivered: same as R1 (refund or replacement, same reporting windows).
R3  Change-of-mind returns (size, fit, no longer needed) are allowed within the category window:
    apparel 30 days, home 10 days, books 10 days, electronics 7 days. Refund after the item is picked up.
R4  Non-returnable categories: grocery and personalised items cannot be returned for change of mind.
R5  Delayed delivery (measured from the promised date):
    up to 3 days late -> apologise and share tracking, no compensation yet;
    4 to 7 days late  -> refund the shipping fee and give a Rs 100 coupon;
    more than 7 days  -> customer may cancel for a full refund.
R6  Goodwill exception: Gold-tier customers up to 3 days outside a return window may be granted an
    exception, but ONLY with manual approval.
R7  Item marked delivered but customer says not received -> courier investigation required before any refund.
R8  Refunds go to the original payment method in 5-7 business days. Cash-on-delivery orders are refunded
    by bank transfer or store credit.
R9  Auto-approval limit: refunds up to Rs 10,000 may be approved automatically. Anything larger needs manual approval.
"""

# The same policy as numbers a program can use
RETURN_WINDOW_DAYS = {"apparel": 30, "home": 10, "books": 10, "electronics": 7}
NON_RETURNABLE = {"grocery", "personalised"}
DAMAGE_WINDOW_DAYS = {"default": 7, "grocery": 2}
PHOTO_REQUIRED_ABOVE_INR = 5000
AUTO_APPROVAL_LIMIT_INR = 10000
GOLD_GRACE_DAYS = 3
DELAY_COUPON_INR = 100

### 2b: Policy rules as Python functions

`check_eligibility()` answers *"what does the policy allow for this order and this issue?"*

`evaluate_escalation()` answers *"does a human need to see this, and how urgently?"* These are our
**escalation rules**, written as clear thresholds so escalation is a deliberate decision:

| Rule | Trigger | Priority | Routed to |
|---|---|---|---|
| ESC_SAFETY | Message mentions fire, smoke, burns, shock or injury | **P1** (urgent) | Trust & Safety |
| ESC_LEGAL | Message mentions consumer court, lawyer, legal notice, police | **P1** | Legal |
| ESC_FRAUD_HISTORY | Customer has 3 or more refund claims in 90 days | **P2** | Fraud & Risk |
| ESC_NOT_RECEIVED | Order marked delivered but customer says not received | **P2** | Fraud & Risk |
| ESC_HIGH_VALUE | Refund above ₹10,000 (auto-approval limit) | **P3** | Refund Approvals |
| ESC_POLICY_EXCEPTION | Gold customer within 3 days outside the window | **P3** | Refund Approvals |
| ESC_LOW_CONFIDENCE | Agent 1's confidence below 0.70 | **P4** | Senior Support |

If none fire, the case is **auto-resolved**, or **denied** when the policy says no, or marked
**needs info** when we can't find the order.

In [ ]:
def _days_between(earlier: str, later: date = REFERENCE_DATE) -> int:
    return (later - date.fromisoformat(earlier)).days


def get_order(order_id: Optional[str]) -> Optional[dict]:
    if not order_id:
        return None
    return ORDERS.get(order_id.strip().upper())


def check_eligibility(order_id: Optional[str], issue_type: str) -> dict:
    """Apply the refund policy to one order + issue. Pure Python: same input, same answer."""
    order = get_order(order_id)
    result = {
        "order_id": order_id, "eligible": False, "rule_id": "", "rule_summary": "",
        "allowed_resolutions": [], "max_refund_inr": 0, "conditions": [],
        "exception_possible": False, "investigation_required": False,
        "days_since_delivery": None, "days_late": None,
    }
    if order is None:
        result.update(rule_id="R0", rule_summary="Order not found. Ask the customer for a valid order ID.",
                      allowed_resolutions=["REQUEST_INFO"])
        return result

    cat, price = order["category"], order["price_inr"]
    delivered = order["status"] == "DELIVERED"
    if delivered:
        result["days_since_delivery"] = _days_between(order["delivered_on"])
    days = result["days_since_delivery"]

    # A "not received" claim on an order that isn't delivered yet is really a delay
    if issue_type == "NOT_RECEIVED" and not delivered:
        issue_type = "DELAYED_DELIVERY"

    if issue_type in ("DAMAGED_PRODUCT", "WRONG_ITEM"):
        window = DAMAGE_WINDOW_DAYS.get(cat, DAMAGE_WINDOW_DAYS["default"])
        rule = "R1" if issue_type == "DAMAGED_PRODUCT" else "R2"
        if days is not None and days <= window:
            conditions = ["Photo evidence of the item"] if price > PHOTO_REQUIRED_ABOVE_INR else []
            result.update(eligible=True, rule_id=rule, max_refund_inr=price, conditions=conditions,
                          allowed_resolutions=["FULL_REFUND", "REPLACEMENT"],
                          rule_summary=f"Reported {days} day(s) after delivery, within the {window}-day window.")
        else:
            result.update(rule_id=rule, allowed_resolutions=["DENY"],
                          rule_summary=f"Reported {days} day(s) after delivery, outside the {window}-day window.")

    elif issue_type == "RETURN_REQUEST":
        if cat in NON_RETURNABLE:
            result.update(rule_id="R4", allowed_resolutions=["DENY"],
                          rule_summary=f"'{cat}' items cannot be returned for change of mind.")
        else:
            window = RETURN_WINDOW_DAYS.get(cat, 7)
            if days is not None and days <= window:
                result.update(eligible=True, rule_id="R3", max_refund_inr=price,
                              allowed_resolutions=["RETURN_AND_REFUND", "EXCHANGE"],
                              conditions=["Item unused, with original tags and packaging"],
                              rule_summary=f"{days} day(s) since delivery, within the {window}-day {cat} window.")
            elif days is not None and order["tier"] == "Gold" and days - window <= GOLD_GRACE_DAYS:
                result.update(rule_id="R6", exception_possible=True, max_refund_inr=price,
                              allowed_resolutions=["RETURN_AND_REFUND", "EXCHANGE"],
                              rule_summary=f"{days - window} day(s) outside the {window}-day window; Gold tier "
                                           f"qualifies for a goodwill exception with manual approval.")
            else:
                result.update(rule_id="R3", allowed_resolutions=["DENY"],
                              rule_summary=f"{days} day(s) since delivery, outside the {window}-day {cat} window.")

    elif issue_type == "DELAYED_DELIVERY":
        if delivered:
            result.update(rule_id="R5", allowed_resolutions=["INFORM"],
                          rule_summary="Order already shows as delivered; no delay compensation applies.")
        else:
            late = _days_between(order["expected_by"])
            result["days_late"] = late
            if late <= 3:
                result.update(eligible=True, rule_id="R5", allowed_resolutions=["INFORM"],
                              rule_summary=f"{max(late, 0)} day(s) late: apologise and share tracking.")
            elif late <= 7:
                result.update(eligible=True, rule_id="R5", allowed_resolutions=["COMPENSATION"],
                              max_refund_inr=order["shipping_fee_inr"],
                              conditions=[f"Also issue a Rs {DELAY_COUPON_INR} coupon"],
                              rule_summary=f"{late} days late: refund shipping fee and give a Rs {DELAY_COUPON_INR} coupon.")
            else:
                result.update(eligible=True, rule_id="R5", allowed_resolutions=["CANCEL_AND_REFUND", "INFORM"],
                              max_refund_inr=price,
                              rule_summary=f"{late} days late: customer may cancel for a full refund.")

    elif issue_type == "NOT_RECEIVED":
        result.update(rule_id="R7", investigation_required=True, max_refund_inr=price,
                      allowed_resolutions=["INVESTIGATE"],
                      rule_summary="Marked delivered but reported not received: courier investigation required first.")

    elif issue_type == "REFUND_STATUS":
        result.update(eligible=True, rule_id="R8", allowed_resolutions=["INFORM"],
                      rule_summary="Refunds reach the original payment method in 5-7 business days "
                                   "(COD: bank transfer or store credit).")
    else:
        result.update(rule_id="NONE", allowed_resolutions=["REQUEST_INFO"],
                      rule_summary="No specific policy rule matches; ask the customer for details.")

    if order["payment"] == "COD" and result["max_refund_inr"] > 0:
        result["conditions"].append("COD order: refund by bank transfer or store credit")
    return result


SAFETY_WORDS = ["fire", "smoke", "smoking", "burn", "explode", "exploded", "shock", "injur", "hurt", "spark"]
LEGAL_WORDS = ["consumer court", "consumer forum", "lawyer", "legal action", "legal notice", "police", "sue you"]

PRIORITY_ROUTE = {
    "P1": "Trust & Safety / Legal team",
    "P2": "Fraud & Risk team",
    "P3": "Refund Approvals team",
    "P4": "Senior Support agent",
}


def evaluate_escalation(order_id: Optional[str], issue_type: str, refund_amount_inr: float,
                        confidence: float, customer_message: str) -> dict:
    """Deterministic escalation rules. Returns which rules fired, the priority and the final status."""
    order = get_order(order_id)
    eligibility = check_eligibility(order_id, issue_type)
    text = (customer_message or "").lower()
    fired = []  # list of (rule_id, priority, reason)

    if any(w in text for w in SAFETY_WORDS):
        fired.append(("ESC_SAFETY", "P1", "Possible product safety hazard or injury mentioned"))
    if any(w in text for w in LEGAL_WORDS):
        fired.append(("ESC_LEGAL", "P1", "Customer mentions legal action"))
    if order and order["refund_claims_90d"] >= 3:
        fired.append(("ESC_FRAUD_HISTORY", "P2", f"{order['refund_claims_90d']} refund claims in the last 90 days"))
    if eligibility["investigation_required"]:
        fired.append(("ESC_NOT_RECEIVED", "P2", "Marked delivered but reported not received"))
    if refund_amount_inr > AUTO_APPROVAL_LIMIT_INR:
        fired.append(("ESC_HIGH_VALUE", "P3", f"Refund Rs {refund_amount_inr:,.0f} exceeds auto-approval limit of Rs {AUTO_APPROVAL_LIMIT_INR:,}"))
    if eligibility["exception_possible"]:
        fired.append(("ESC_POLICY_EXCEPTION", "P3", "Goodwill policy exception needs manual approval"))
    if confidence < 0.70 and order is not None:
        fired.append(("ESC_LOW_CONFIDENCE", "P4", f"Issue classification confidence {confidence:.2f} is below 0.70"))

    if fired:
        priority = min(p for _, p, _ in fired)          # "P1" < "P2" < ... so min = most urgent
        status, escalate = "ESCALATED", True
    elif order is None:
        priority, status, escalate = "NONE", "NEEDS_INFO", False
    elif not eligibility["eligible"]:
        priority, status, escalate = "NONE", "DENIED", False
    else:
        priority, status, escalate = "NONE", "AUTO_RESOLVED", False

    return {
        "escalate": escalate,
        "priority": priority,
        "route_to": PRIORITY_ROUTE.get(priority, "None: handled automatically"),
        "final_status": status,
        "triggered_rules": [r for r, _, _ in fired],
        "reasons": [why for _, _, why in fired],
        "policy_rule_id": eligibility["rule_id"],
    }

### 2c: Test the rules before adding any AI

A good habit: check the deterministic parts work **before** wiring in the LLM. If something goes
wrong later, you'll know the problem is in the agents, not your rules. This cell uses no API calls.

In [ ]:
quick_checks = [
    ("ORD1001", "DAMAGED_PRODUCT", 2499,  0.95, "Headphones arrived broken"),
    ("ORD1004", "DAMAGED_PRODUCT", 62000, 0.95, "Laptop screen cracked"),
    ("ORD1006", "DAMAGED_PRODUCT", 1499,  0.95, "Power bank started smoking"),
    ("ORD1007", "RETURN_REQUEST",  0,     0.90, "Don't want the fruit any more"),
    ("ORD1008", "RETURN_REQUEST",  4599,  0.90, "Shoes don't fit"),
]
rows = []
for oid, issue, amt, conf, msg in quick_checks:
    e = check_eligibility(oid, issue)
    x = evaluate_escalation(oid, issue, amt, conf, msg)
    rows.append({"order": oid, "issue": issue, "policy_rule": e["rule_id"], "eligible": e["eligible"],
                 "status": x["final_status"], "priority": x["priority"], "rules_fired": ", ".join(x["triggered_rules"])})
pd.DataFrame(rows)

---
## Step 3: Tools: how agents reach the knowledge layer

A **tool** is a Python function an agent is allowed to call. The `@tool` decorator turns a normal
function into one. The agent reads the **docstring** to decide when and how to use it, so write
docstrings as instructions.

Each agent only gets the tools it needs (least privilege):
- Agent 1 → `lookup_order`
- Agent 2 → `check_refund_policy`
- Agent 3 → no tools (it reasons over what it was handed)
- Agent 4 → `run_escalation_rules`

In [ ]:
@tool("lookup_order")
def lookup_order(order_id: str) -> str:
    """Look up an order in ShopKart's order database by its order ID (format ORD followed by 4 digits,
    e.g. ORD1001). Returns the order details as JSON, or an error if the order does not exist."""
    order = get_order(order_id)
    if order is None:
        return json.dumps({"found": False, "order_id": order_id, "error": "No order with this ID"})
    return json.dumps({"found": True, "order_id": order_id.strip().upper(), **order,
                       "today": REFERENCE_DATE.isoformat()})


@tool("check_refund_policy")
def check_refund_policy(order_id: str, issue_type: str) -> str:
    """Apply ShopKart's official refund/return policy to an order. issue_type must be one of:
    DAMAGED_PRODUCT, WRONG_ITEM, RETURN_REQUEST, DELAYED_DELIVERY, NOT_RECEIVED, REFUND_STATUS, OTHER.
    Returns eligibility, the policy rule applied, allowed resolutions, the maximum refund and conditions.
    This result is authoritative: do not override it."""
    return json.dumps(check_eligibility(order_id, issue_type))


@tool("run_escalation_rules")
def run_escalation_rules(order_id: str, issue_type: str, refund_amount_inr: float,
                         confidence: float, customer_message: str) -> str:
    """Run ShopKart's mandatory escalation rules. Pass the order ID (or 'NONE'), the issue type, the
    proposed refund amount in rupees, the issue classification confidence (0-1) and the customer's
    original message word for word. Returns whether to escalate, the priority, the team to route to,
    the final status and which rules fired. This result is authoritative: do not override it."""
    oid = None if (order_id or "").upper() in ("", "NONE", "NULL", "N/A") else order_id
    return json.dumps(evaluate_escalation(oid, issue_type, float(refund_amount_inr),
                                          float(confidence), customer_message))

---
## Step 4: Handoff formats: what exactly gets passed between agents

This is the most important idea in the project. If Agent 1 hands over a paragraph of free text,
Agent 2 has to guess what it means. Instead, every agent produces a **Pydantic model**: a fixed set of
typed fields. CrewAI makes the LLM fill these fields, then passes them to the next agent.

`Literal[...]` restricts a field to a fixed list of values, so the LLM can't invent a new category
like "kinda broken".

In [ ]:
IssueType = Literal["DAMAGED_PRODUCT", "WRONG_ITEM", "RETURN_REQUEST", "DELAYED_DELIVERY",
                    "NOT_RECEIVED", "REFUND_STATUS", "OTHER"]


class IssueReport(BaseModel):
    """Agent 1 → Agent 2"""
    order_id: Optional[str] = Field(description="Order ID from the message, e.g. ORD1001, or null if none was given")
    order_found: bool = Field(description="True only if lookup_order found the order")
    issue_type: IssueType
    customer_request: Literal["REFUND", "REPLACEMENT", "RETURN", "EXCHANGE", "CANCEL", "INFORMATION", "UNCLEAR"]
    sentiment: Literal["CALM", "FRUSTRATED", "ANGRY"]
    safety_or_legal_concern: bool = Field(description="True if the message mentions injury, fire, smoke, or legal action")
    confidence: float = Field(ge=0, le=1, description="How sure you are about issue_type (0 to 1)")
    summary: str = Field(description="One or two sentences: what happened and what the customer wants")


class PolicyAssessment(BaseModel):
    """Agent 2 → Agent 3"""
    eligible: bool
    rule_id: str = Field(description="Policy rule applied, e.g. R1")
    allowed_resolutions: List[str]
    max_refund_inr: float
    conditions: List[str]
    exception_possible: bool
    investigation_required: bool
    explanation: str = Field(description="Plain-English explanation of why the policy gives this result")


class ResolutionPlan(BaseModel):
    """Agent 3 → Agent 4"""
    action: Literal["FULL_REFUND", "REPLACEMENT", "RETURN_AND_REFUND", "EXCHANGE", "COMPENSATION",
                    "CANCEL_AND_REFUND", "INVESTIGATE", "INFORM", "REQUEST_INFO", "DENY"]
    refund_amount_inr: float = Field(description="Refund being proposed, never above max_refund_inr; 0 if none")
    customer_reply_draft: str = Field(description="Polite reply to the customer, 60-120 words")
    internal_notes: str = Field(description="Short note for the support team")


class EscalationDecision(BaseModel):
    """Agent 4 → final output"""
    escalate: bool
    priority: Literal["P1", "P2", "P3", "P4", "NONE"]
    triggered_rules: List[str]
    route_to: str
    final_status: Literal["AUTO_RESOLVED", "ESCALATED", "DENIED", "NEEDS_INFO"]
    final_customer_message: str = Field(description="The message actually sent to the customer")
    justification: str = Field(description="One or two sentences explaining the decision")

---
## Step 5: The agents: roles, goals, backstories

Each agent has **one tight responsibility** (no "do-everything" agents). `role`, `goal` and
`backstory` become the agent's system prompt.

`allow_delegation=False` stops agents from handing work to each other on their own. **We** decide the
order of handoffs, which keeps the workflow predictable and easy to explain.

We wrap everything in functions (`build_agents`, `build_tasks`, `build_crew`) so each customer message
gets a fresh crew, and one case can't leak into the next.

In [ ]:
def build_agents():
    issue_agent = Agent(
        role="Order Issue Identification Specialist",
        goal="Read the customer's message, find their order and classify the issue precisely, "
             "with an honest confidence score.",
        backstory="You are the first line of ShopKart support. You never guess order details: you always "
                  "look the order up. You only classify; you never promise refunds or quote policy.",
        tools=[lookup_order],
        llm=llm, allow_delegation=False, max_iter=5, verbose=False,
    )

    policy_agent = Agent(
        role="Refund & Returns Policy Analyst",
        goal="Determine exactly what ShopKart's policy allows for the classified issue.",
        backstory="You are ShopKart's policy expert. You always run check_refund_policy and treat its result "
                  "as the official answer. You explain it clearly but never bend it. Policy for reference:\n"
                  + REFUND_POLICY_TEXT,
        tools=[check_refund_policy],
        llm=llm, allow_delegation=False, max_iter=5, verbose=False,
    )

    resolution_agent = Agent(
        role="Customer Resolution Specialist",
        goal="Choose the best resolution that the policy allows and write a warm, clear reply to the customer.",
        backstory="You turn policy decisions into good customer outcomes. You only choose from the allowed "
                  "resolutions, never exceed the maximum refund and never invent compensation. "
                  "You write empathetically, in simple English.",
        tools=[],
        llm=llm, allow_delegation=False, max_iter=3, verbose=False,
    )

    escalation_agent = Agent(
        role="Risk & Escalation Officer",
        goal="Decide whether the case can be closed automatically or needs human approval, using the "
             "mandatory escalation rules.",
        backstory="You are the final checkpoint before anything reaches the customer. You always run "
                  "run_escalation_rules and follow its result exactly. You never downgrade an escalation. "
                  "When a case is escalated, you make sure the customer is not promised anything a human "
                  "still has to approve.",
        tools=[run_escalation_rules],
        llm=llm, allow_delegation=False, max_iter=5, verbose=False,
    )
    return issue_agent, policy_agent, resolution_agent, escalation_agent

---
## Step 6: The tasks and the handoffs

A **task** is a unit of work given to one agent. Three settings matter most here:
- `description`: the instructions. `{customer_message}` is filled in at kickoff.
- `output_pydantic`: forces the output into the handoff format from Step 4.
- `context=[...]`: **the handoff.** It passes the outputs of earlier tasks into this one.

```
identify_task ──► policy_task ──► resolution_task ──► escalation_task
      └───────────────┴──────────────────┴─────────────────► (escalation sees all three)
```

In [ ]:
def build_tasks(issue_agent, policy_agent, resolution_agent, escalation_agent):
    identify_task = Task(
        description=(
            "A customer wrote to ShopKart support:\n\n"
            "\"\"\"{customer_message}\"\"\"\n\n"
            "1. Extract the order ID if one is present (format ORD + 4 digits).\n"
            "2. If there is an order ID, call lookup_order to fetch the real order details.\n"
            "3. Classify the issue as exactly one of: DAMAGED_PRODUCT, WRONG_ITEM, RETURN_REQUEST "
            "(change of mind, size or fit), DELAYED_DELIVERY, NOT_RECEIVED, REFUND_STATUS, OTHER.\n"
            "4. Flag safety_or_legal_concern if the customer mentions injury, fire, smoke, burning, "
            "electric shock or legal action.\n"
            "5. Give a confidence score. Use below 0.7 when the message is vague or could fit several issue types."
        ),
        expected_output="An IssueReport with the order ID, issue type, customer request, sentiment, flags, confidence and summary.",
        agent=issue_agent,
        output_pydantic=IssueReport,
    )

    policy_task = Task(
        description=(
            "Using the IssueReport from the previous step, call check_refund_policy with its order_id and "
            "issue_type (use 'NONE' as the order_id if there is none). Report the result faithfully: "
            "eligibility, rule ID, allowed resolutions, maximum refund, conditions, whether an exception is "
            "possible and whether investigation is required. Then explain it in plain English."
        ),
        expected_output="A PolicyAssessment that matches the check_refund_policy result exactly.",
        agent=policy_agent,
        context=[identify_task],                     # handoff: Agent 1 → Agent 2
        output_pydantic=PolicyAssessment,
    )

    resolution_task = Task(
        description=(
            "Customer message: \"\"\"{customer_message}\"\"\"\n\n"
            "Using the IssueReport and PolicyAssessment, choose ONE action from allowed_resolutions, "
            "preferring what the customer asked for when it is allowed. Rules:\n"
            "- refund_amount_inr must not exceed max_refund_inr (use 0 if no money is refunded).\n"
            "- If not eligible, choose DENY and explain kindly why, mentioning any alternative.\n"
            "- If investigation_required is true, choose INVESTIGATE.\n"
            "- If exception_possible is true, propose the refund but say it is subject to approval.\n"
            "- Mention any conditions (for example photo evidence).\n"
            "Write the customer reply addressing the customer by first name if known."
        ),
        expected_output="A ResolutionPlan with an action, refund amount, customer reply draft and internal notes.",
        agent=resolution_agent,
        context=[identify_task, policy_task],        # handoff: Agents 1 + 2 → Agent 3
        output_pydantic=ResolutionPlan,
    )

    escalation_task = Task(
        description=(
            "Customer message: \"\"\"{customer_message}\"\"\"\n\n"
            "Call run_escalation_rules with: the order_id and issue_type from the IssueReport, the "
            "refund_amount_inr from the ResolutionPlan, the confidence from the IssueReport, and the "
            "customer's original message above, copied word for word.\n"
            "Copy escalate, priority, triggered_rules, route_to and final_status from the tool result exactly.\n"
            "Write final_customer_message:\n"
            "- If NOT escalated, use the ResolutionPlan's customer_reply_draft (light edits only).\n"
            "- If escalated, do NOT promise the refund or outcome. Acknowledge the issue, say a specialist "
            "team will review it (P1 within 4 hours, P2 or P3 within 24 hours, P4 within 48 hours), and "
            "for safety issues tell the customer to stop using the product."
        ),
        expected_output="An EscalationDecision that matches the run_escalation_rules result.",
        agent=escalation_agent,
        context=[identify_task, policy_task, resolution_task],   # handoff: Agents 1-3 → Agent 4
        output_pydantic=EscalationDecision,
    )
    return identify_task, policy_task, resolution_task, escalation_task

---
## Step 7: Assemble the crew and run one case

`Process.sequential` runs the tasks in order: each one waits for the one before it.

`run_case()` sends one customer message through the crew and collects every agent's structured output,
so we can inspect each handoff. Run the demo with `verbose=True` to watch the agents think and call tools.

In [ ]:
def build_crew(verbose: bool = False) -> Crew:
    agents = build_agents()
    tasks = build_tasks(*agents)
    return Crew(agents=list(agents), tasks=list(tasks), process=Process.sequential, verbose=verbose)


def run_case(customer_message: str, verbose: bool = False) -> dict:
    crew = build_crew(verbose=verbose)
    result = crew.kickoff(inputs={"customer_message": customer_message})
    issue, policy, plan, decision = (t.pydantic for t in result.tasks_output)
    return {"message": customer_message, "issue": issue, "policy": policy, "plan": plan, "decision": decision}


def show_case(case: dict) -> None:
    i, p, r, d = case["issue"], case["policy"], case["plan"], case["decision"]
    print("CUSTOMER  :", case["message"])
    print(f"AGENT 1   : {i.issue_type} | order {i.order_id} (found={i.order_found}) | wants {i.customer_request} "
          f"| confidence {i.confidence:.2f} | safety/legal={i.safety_or_legal_concern}")
    print(f"AGENT 2   : rule {p.rule_id} | eligible={p.eligible} | allowed {p.allowed_resolutions} "
          f"| max ₹{p.max_refund_inr:,.0f}")
    print(f"AGENT 3   : {r.action} | refund ₹{r.refund_amount_inr:,.0f}")
    print(f"AGENT 4   : {d.final_status} | escalate={d.escalate} | {d.priority} | {d.route_to} | rules {d.triggered_rules}")
    print("REPLY     :", d.final_customer_message)
    print("-" * 100)

In [ ]:
demo = run_case("Hi, my order ORD1001 arrived yesterday but the left side of the headphones doesn't work. "
                "I'd like a refund please.", verbose=True)
show_case(demo)

---
## Step 8: Sample inputs and evaluation

Ten realistic messages, including edge cases that **must** escalate and two that must **not**
(testing for false positives). `expected_status` is what a correct system should decide.

- **False positive** = escalated when it shouldn't have been (wastes human time)
- **False negative** = not escalated when it should have been (risky: money or safety)

In [ ]:
SAMPLE_CASES = [
    ("C01", "Hi, my order ORD1001 arrived 3 days ago but the left earcup of the headphones doesn't work. I'd like a refund.", "AUTO_RESOLVED"),
    ("C02", "The kurta from order ORD1002 is too tight. Can I return it?", "AUTO_RESOLVED"),
    ("C03", "Where is my smart watch? Order ORD1003 was supposed to arrive on 12 October and it's still not here!", "AUTO_RESOLVED"),
    ("C04", "My new laptop (ORD1004) arrived with a cracked screen. I want my money back.", "ESCALATED"),
    ("C05", "Order ORD1005 says delivered but I never got the phone. Refund me now.", "ESCALATED"),
    ("C06", "The power bank from ORD1006 started smoking while charging and burned my hand. This is dangerous.", "ESCALATED"),
    ("C07", "I ordered a fruit basket (ORD1007) but I've changed my mind. Please take it back and refund me.", "DENIED"),
    ("C08", "I'm a Gold member. The running shoes from ORD1008 don't fit. I know it's been a month but can you still accept a return?", "ESCALATED"),
    ("C09", "You sent me the wrong book for ORD1009! This is the third time something's gone wrong. Fix it or I'm going to consumer court.", "ESCALATED"),
    ("C10", "my thing came and its not right, do something", "NEEDS_INFO"),
]

In [ ]:
results = []
for case_id, message, expected in SAMPLE_CASES:
    print(f"\n### {case_id}")
    try:
        case = run_case(message)
        show_case(case)
        d = case["decision"]
        results.append({
            "case": case_id, "issue_type": case["issue"].issue_type, "policy_rule": case["policy"].rule_id,
            "action": case["plan"].action, "refund_inr": case["plan"].refund_amount_inr,
            "status": d.final_status, "expected": expected, "priority": d.priority,
            "rules_fired": ", ".join(d.triggered_rules), "reply": d.final_customer_message,
        })
    except Exception as err:   # one failing case shouldn't stop the whole batch
        print("ERROR:", err)
        results.append({"case": case_id, "status": "ERROR", "expected": expected, "rules_fired": str(err)})

results_df = pd.DataFrame(results)
results_df["correct"] = results_df["status"] == results_df["expected"]
results_df[["case", "issue_type", "policy_rule", "action", "refund_inr", "status", "expected", "priority", "rules_fired", "correct"]]

### Escalation scorecard

In [ ]:
should_escalate = results_df["expected"] == "ESCALATED"
did_escalate = results_df["status"] == "ESCALATED"

print(f"Overall accuracy          : {results_df['correct'].mean():.0%} ({results_df['correct'].sum()}/{len(results_df)})")
print(f"Correct escalations       : {(should_escalate & did_escalate).sum()} / {should_escalate.sum()}")
print(f"False positives (over-esc): {(~should_escalate & did_escalate).sum()}  -> {list(results_df.loc[~should_escalate & did_escalate, 'case'])}")
print(f"False negatives (missed)  : {(should_escalate & ~did_escalate).sum()}  -> {list(results_df.loc[should_escalate & ~did_escalate, 'case'])}")

os.makedirs("outputs", exist_ok=True)
results_df.to_csv("outputs/results.csv", index=False)
print("\nSaved to outputs/results.csv (use it for the Sample inputs & outputs section of your write-up)")

---
## Step 9: Try your own messages and refine

Change the message below and re-run. Things to try:
- Remove the order ID from a message: does it become `NEEDS_INFO`?
- Write a vague message about a real order: does confidence drop below 0.7 and escalate?
- Change `AUTO_APPROVAL_LIMIT_INR` in Step 2 to 1000 and re-run Step 8. How many cases escalate now?

Those experiments are what the brief means by *"use your tests to refine handoff formats, policy
clarity and escalation decisions"*. Note what you changed and why in your write-up.

In [ ]:
my_case = run_case("Order ORD1002: the kurta colour looks different from the photo, can I exchange it for another colour?")
show_case(my_case)